## 03. Feature Engineering: Classical Representation

U ovom notebooku proteinske sekvence se transformišu u numeričke reprezentacije
pogodne za primjenu klasičnih metoda mašinskog učenja.

Razmatrane su tri grupe obilježja:

1. **AAC-CV (Amino Acid Composition, count vectorizer)** — frekvencija 20 standardnih aminokiselina,
2. **TF-IDF** — karakter-n-gram reprezentacija proteinskih sekvenci,
3. **Physicochemical Features (PH)** — fizičko-hemijske osobine proteinskih sekvenci.


### Uvoz biblioteka

In [1]:
import os
import ast
import joblib
import numpy as np
import pandas as pd

from Bio.SeqUtils.ProtParam import ProteinAnalysis

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, Normalizer, StandardScaler, MultiLabelBinarizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD

### Učitavanje podataka 

In [2]:
DATA_DIR = "../data/processed"
FEATURES_DIR = "../data/features"
os.makedirs(FEATURES_DIR, exist_ok=True)

DATASET_PATH = os.path.join(DATA_DIR, "dataset_final.csv")
PHYSCHEM_PATH = os.path.join(DATA_DIR, "physicochemical_features.csv")

In [3]:
dataset = pd.read_csv(DATASET_PATH)

dataset['label_list'] = dataset['label'].apply(ast.literal_eval)

print(f"Ukupan broj proteina: {len(dataset)}")
dataset.head()

Ukupan broj proteina: 7043


,Sequence,Entry,label,num_classes,is_multilabel,Hydrolase,Receptor,Structural protein,Transcription factor,Transport protein,label_list
0,DKQLDADVSPKPTIFLPSIAETKLQKAGTYLCLLEKFFPDIIKIHW...,P03986,['Receptor'],1,False,0,1,0,0,0,[Receptor]
1,MAAAAAAAAAVGVRLRDCCSRGAVLLLFFSLSPRPPAAAAWLLGLR...,Q9NRU3,['Transport protein'],1,False,0,0,0,0,1,[Transport protein]
2,MAAAAAALSGAGTPPAGGGAGGGGAGGGGSPPGGWAVARLEGREFE...,Q01167,['Transcription factor'],1,False,0,0,0,1,0,[Transcription factor]
3,MAAAAAEEGMEPRALQYEQTLMYGRYTQDLGAFAKEEAARIRLGGP...,P51788,['Transport protein'],1,False,0,0,0,0,1,[Transport protein]
4,MAAAAALRAPAQSSVTFEDVAVNFSLEEWSLLNEAQRCLYRDVMLE...,Q9NXT0,['Transcription factor'],1,False,0,0,0,1,0,[Transcription factor]


### Fizičko-hemijske osobine

In [4]:
def extract_physicochemical(sequence):
    analysed_seq = ProteinAnalysis(sequence)
    return {
        "MW": analysed_seq.molecular_weight(),
        "pI": analysed_seq.isoelectric_point(),
        "GRAVY": analysed_seq.gravy(),
        "Aromaticity": analysed_seq.aromaticity(),
        "Instability": analysed_seq.instability_index()
    }

if os.path.exists(PHYSCHEM_PATH):
    print("Fizicko-hemijske osobine vec postoje, ucitavanje...")
    physchem_df = pd.read_csv(PHYSCHEM_PATH, index_col="Entry")
else:
    print("Racunam fizicko-hemijske osobine...")
    physchem_df = dataset["Sequence"].apply(lambda seq: pd.Series(extract_physicochemical(seq)))
    physchem_df.index = dataset["Entry"]
    physchem_df.to_csv(PHYSCHEM_PATH)
    print(f"Sacuvano u '{PHYSCHEM_PATH}'")

physchem_df.head()

Fizicko-hemijske osobine vec postoje, ucitavanje...


,MW,pI,GRAVY,Aromaticity,Instability
Entry,,,,,
P03986,21697.6278,5.954646,-0.401058,0.095238,29.608995
Q9NRU3,104349.3927,5.910255,-0.242376,0.072555,53.433470
Q01167,69061.1230,9.564739,-0.361061,0.043939,57.529864
P51788,98534.3057,8.700734,0.132517,0.086860,50.363586
Q9NXT0,46412.7668,9.138410,-0.897761,0.082090,61.421642


### Priprema SC podskupa

In [5]:
dataset_sc = dataset[dataset["label_list"].apply(len) == 1].copy()
dataset_sc["label_sc"] = dataset_sc["label_list"].apply(lambda x: x[0])

print(f"Broj proteina u SC podskupu {len(dataset_sc)}")
print(dataset_sc["label_sc"].value_counts())

Broj proteina u SC podskupu 6663
label_sc
Hydrolase               2204
Receptor                1378
Transcription factor    1346
Transport protein       1023
Structural protein       712
Name: count, dtype: int64


#### Podjela na ulazne i ciljne atribute

In [6]:
X_entries = dataset_sc["Entry"].values
y_raw = dataset_sc["label_sc"].values

print(f"Broj sekvenci: {len(X_entries)}")
print(f"Broj oznaka: {len(y_raw)}")

Broj sekvenci: 6663
Broj oznaka: 6663


#### Kodiranje ciljne varijable (Label Encoding)

In [7]:
le = LabelEncoder()
y_encoded = le.fit_transform(y_raw)

print("Mapiranje klasa:")
for num, label in enumerate(le.classes_):
    print(f"{num} -> {label}")

joblib.dump(le, os.path.join(FEATURES_DIR, "sc_label_encoder.pkl"))

Mapiranje klasa:
0 -> Hydrolase
1 -> Receptor
2 -> Structural protein
3 -> Transcription factor
4 -> Transport protein


['../data/features\\sc_label_encoder.pkl']

#### Train/test split

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

entries_sc_train, entries_sc_test, y_sc_train, y_sc_test = train_test_split(
    X_entries, y_encoded,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_encoded,
)

print(f"Train skup: {len(entries_sc_train)} proteina (80%)")
print(f"Test skup:  {len(entries_sc_test)} proteina (20%)")

# Čuvamo Entry liste i encoded labele zajedno — jedan izvor (source of truth) za split
pd.DataFrame({"Entry": entries_sc_train, "label_encoded": y_sc_train}).to_csv(
    os.path.join(FEATURES_DIR, "sc_train_entries.csv"), index=False
)
pd.DataFrame({"Entry": entries_sc_test, "label_encoded": y_sc_test}).to_csv(
    os.path.join(FEATURES_DIR, "sc_test_entries.csv"), index=False
)

Train skup: 5330 proteina (80%)
Test skup:  1333 proteina (20%)


### Priprema MC podskupa

In [9]:
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

mlb = MultiLabelBinarizer()
y_mc = mlb.fit_transform(dataset["label_list"])
label_columns = mlb.classes_

joblib.dump(mlb, os.path.join(FEATURES_DIR, "mc_label_binarizer.pkl"))

mc_label_df = pd.DataFrame(y_mc, columns=label_columns, index=dataset["Entry"])
mc_label_df.to_csv(os.path.join(FEATURES_DIR, "mc_labels.csv"))

splitter = MultilabelStratifiedShuffleSplit(
    n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

train_idx, test_idx = next(splitter.split(dataset["Entry"], y_mc))

entries_mc_train = dataset["Entry"].iloc[train_idx]
entries_mc_test = dataset["Entry"].iloc[test_idx]

print(f"Train skup: {len(entries_mc_train)} proteina (80%)")
print(f"Test skup:  {len(entries_mc_test)} proteina (20%)")

pd.Series(entries_mc_train).to_csv(os.path.join(FEATURES_DIR, "mc_train_entries.csv"), index=False)
pd.Series(entries_mc_test).to_csv(os.path.join(FEATURES_DIR, "mc_test_entries.csv"), index=False)


Train skup: 5643 proteina (80%)
Test skup:  1400 proteina (20%)


### AAC CountVectorizer

In [10]:
AMINO_ACIDS = ['A','C','D','E','F','G','H','I','K','L',
               'M','N','P','Q','R','S','T','V','W','Y']
cv_aac = CountVectorizer(analyzer='char', vocabulary=AMINO_ACIDS, lowercase=False)
normalizer = Normalizer(norm='l1')

aac_matrix = normalizer.transform(cv_aac.fit_transform(dataset["Sequence"]))
aac_df = pd.DataFrame(aac_matrix.toarray(), columns=AMINO_ACIDS, index=dataset["Entry"])

aac_df.to_csv(os.path.join(FEATURES_DIR, "aac_cv_features.csv"))
print(f"AAC-CV dimenzije: {aac_df.shape}")
aac_df.head()

AAC-CV dimenzije: (7043, 20)


,A,C,D,E,F,G,H,I,K,L,M,N,P,Q,R,S,T,V,W,Y
Entry,,,,,,,,,,,,,,,,,,,,
P03986,0.042328,0.031746,0.079365,0.052910,0.037037,0.026455,0.015873,0.074074,0.100529,0.105820,0.015873,0.063492,0.047619,0.031746,0.021164,0.058201,0.095238,0.042328,0.015873,0.042328
Q9NRU3,0.083070,0.021030,0.051525,0.070452,0.037855,0.079916,0.017876,0.028391,0.033649,0.126183,0.012618,0.032597,0.065195,0.030494,0.075710,0.075710,0.057834,0.065195,0.009464,0.025237
Q01167,0.110606,0.006061,0.025758,0.045455,0.021212,0.092424,0.028788,0.046970,0.042424,0.060606,0.012121,0.031818,0.107576,0.057576,0.051515,0.093939,0.068182,0.074242,0.004545,0.018182
P51788,0.102450,0.018931,0.028953,0.062361,0.052339,0.072383,0.017817,0.052339,0.035635,0.110245,0.025612,0.015590,0.060134,0.037862,0.065702,0.073497,0.059020,0.074610,0.015590,0.018931
Q9NXT0,0.042289,0.059701,0.017413,0.097015,0.037313,0.067164,0.074627,0.034826,0.064677,0.069652,0.007463,0.022388,0.032338,0.034826,0.099502,0.114428,0.047264,0.032338,0.004975,0.039801


### TF-IDF (Term Frequency-Inverse Document Frequency)

Transformišemo proteinske sekvence u numeričke vektore koristeći **TF-IDF** statistiku. TF-IDF penalizuje n-grame koji se prečesto pojavljuju u svim klasama, a naglašava one koji su jedinstveni za specifične funkcionalne grupe.

In [11]:
def fit_transform_tfidf_svd(train_sequences, test_sequences, n_components=150, ngram_range=(2,4), max_features=5000):
    tfidf = TfidfVectorizer(
        analyzer='char', 
        ngram_range=ngram_range, 
        max_features=max_features, 
        lowercase=False
    )
    
    svd = TruncatedSVD(n_components=n_components, random_state=RANDOM_STATE)

    train_tfidf = tfidf.fit_transform(train_sequences)
    test_tfidf = tfidf.transform(test_sequences)

    train_svd = svd.fit_transform(train_tfidf)
    test_svd = svd.transform(test_tfidf)

    explained_var = svd.explained_variance_ratio_.sum()
    print(f"SVD objašnjena varijansa ({n_components} komponenti): {explained_var:.3f}")

    return train_svd, test_svd, tfidf, svd

### TF-IDF+SVD za Eksperiment A (SC)

In [12]:
seq_by_entry = dataset.set_index("Entry")["Sequence"]

sc_train_seq = seq_by_entry.loc[entries_sc_train]
sc_test_seq = seq_by_entry.loc[entries_sc_test]

sc_train_tfidf, sc_test_tfidf, tfidf_sc, svd_sc = fit_transform_tfidf_svd(sc_train_seq, sc_test_seq)

# Cuvanje matrica
np.save(os.path.join(FEATURES_DIR, "sc_train_tfidf_svd.npy"), sc_train_tfidf)
np.save(os.path.join(FEATURES_DIR, "sc_test_tfidf_svd.npy"), sc_test_tfidf)
# Cuvanje transformera
joblib.dump(tfidf_sc, os.path.join(FEATURES_DIR, "sc_tfidf_vectorizer.pkl"))
joblib.dump(svd_sc, os.path.join(FEATURES_DIR, "sc_svd.pkl"))

SVD objašnjena varijansa (150 komponenti): 0.331


['../data/features\\sc_svd.pkl']

### TF-IDF+SVD za Eksperiment C (MC)

In [13]:
mc_train_seq = seq_by_entry.loc[entries_mc_train]
mc_test_seq = seq_by_entry.loc[entries_mc_test]

mc_train_tfidf, mc_test_tfidf, tfidf_mc, svd_mc = fit_transform_tfidf_svd(mc_train_seq, mc_test_seq)

# Cuvanje matrica
np.save(os.path.join(FEATURES_DIR, "mc_train_tfidf_svd.npy"), mc_train_tfidf)
np.save(os.path.join(FEATURES_DIR, "mc_test_tfidf_svd.npy"), mc_test_tfidf)
# Cuvanje transformera
joblib.dump(tfidf_mc, os.path.join(FEATURES_DIR, "mc_tfidf_vectorizer.pkl"))
joblib.dump(svd_mc, os.path.join(FEATURES_DIR, "mc_svd.pkl"))

SVD objašnjena varijansa (150 komponenti): 0.327


['../data/features\\mc_svd.pkl']

### Skaliranje fizičko-hemijskih osobina

In [14]:
def scale_physchem(train_entries, test_entries, prefix):
    scaler = StandardScaler()
    train_ph = scaler.fit_transform(physchem_df.loc[train_entries])
    test_ph = scaler.transform(physchem_df.loc[test_entries])

    # Cuvanje matrice
    np.save(os.path.join(FEATURES_DIR, f"{prefix}_train_physchem_scaled.npy"), train_ph)
    np.save(os.path.join(FEATURES_DIR, f"{prefix}_test_physchem_scaled.npy"), test_ph)
    # Cuvanje transformera
    joblib.dump(scaler, os.path.join(FEATURES_DIR, f"{prefix}_physchem_scaler.pkl"))

    return train_ph, test_ph

sc_train_ph, sc_test_ph = scale_physchem(entries_sc_train, entries_sc_test, "sc")
mc_train_ph, mc_test_ph = scale_physchem(entries_mc_train, entries_mc_test, "mc")